# Meta MMS ASR Training for Twi
## Massively Multilingual Speech Model Fine-tuning

Meta's MMS supports 1,100+ languages including Twi. This notebook fine-tunes MMS for improved Twi ASR performance.

In [ ]:
# Installation and imports
!pip install -q transformers datasets torch torchaudio evaluate jiwer tensorboard
!pip install -q accelerate librosa soundfile

import os
import sys
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from dataclasses import dataclass
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Audio processing
import torchaudio
import librosa
import soundfile as sf

# MMS model imports
from transformers import (
    Wav2Vec2ForCTC,
    AutoProcessor,
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict, Audio
import evaluate

# Check environment
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules

print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🎯 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

print(f"\n🌍 Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

# Auto-detect and use correct data path
if IN_COLAB:
    # Mount Google Drive if in Colab
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATA_PATH = Path('/content/drive/MyDrive/voice-lab-data')
    except Exception as e:
        print(f"⚠️ Could not mount Google Drive: {e}")
        print("   Using local Colab storage instead")
        DATA_PATH = Path('/content/voice-lab-data')
        DATA_PATH.mkdir(parents=True, exist_ok=True)
elif IN_KAGGLE:
    DATA_PATH = Path('/kaggle/working/voice-lab-data')
else:
    # Local environment
    DATA_PATH = Path('/opt/kasaai/apps/voice-lab/data')

print(f"📂 Data path: {DATA_PATH}")

# Ensure data path exists
DATA_PATH.mkdir(parents=True, exist_ok=True)

# Check if processed data exists
metadata_path = DATA_PATH / 'metadata/all_datasets_processed.csv'
if metadata_path.exists():
    processed_data = pd.read_csv(metadata_path)
    print(f"✅ Found processed data: {len(processed_data)} samples")
    print(f"   Languages: {processed_data['language'].unique() if 'language' in processed_data.columns else 'N/A'}")
    print(f"   Datasets: {processed_data['dataset'].unique() if 'dataset' in processed_data.columns else 'N/A'}")
else:
    print(f"⚠️ Processed data not found at {metadata_path}")
    print("   Please run notebooks 001 and 002 first to download and process data")
    print("   Or the notebook will create dummy data for testing")

In [ ]:
# Configuration
@dataclass
class MMSConfig:
    """Configuration for MMS training."""
    # Model settings
    model_name: str = "facebook/mms-1b-all"  # MMS 1B parameter model
    language: str = "twi"  # Twi language name for display
    adapter_lang: str = "aka"  # Language adapter code (aka = Akan, which includes Twi)
    
    # Data paths - use global DATA_PATH
    data_dir: Path = DATA_PATH
    output_dir: Path = DATA_PATH.parent / "models" / "mms_twi"
    cache_dir: Path = DATA_PATH.parent / "cache"
    
    # Training parameters
    batch_size: int = 8
    gradient_accumulation_steps: int = 4
    learning_rate: float = 3e-4
    warmup_steps: int = 500
    num_train_epochs: int = 30
    fp16: bool = True
    gradient_checkpointing: bool = True
    
    # Audio settings
    sample_rate: int = 16000
    max_duration: float = 20.0  # MMS works better with shorter segments
    
    # Evaluation
    eval_steps: int = 500
    save_steps: int = 500
    logging_steps: int = 100
    
    # MMS specific
    freeze_feature_encoder: bool = True  # Freeze CNN feature encoder
    mask_time_prob: float = 0.05  # Masked time for training
    layerdrop: float = 0.0  # Layer dropout
    
    def __post_init__(self):
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

config = MMSConfig()
print(f"📋 MMS Configuration loaded")
print(f"   Language: {config.language} (using adapter: {config.adapter_lang})")
print(f"   Data directory: {config.data_dir}")
print(f"   Output directory: {config.output_dir}")

In [ ]:
# Load MMS model with Twi adapter
class MMSModelLoader:
    """Load and configure MMS model for Twi."""
    
    def __init__(self, config: MMSConfig):
        self.config = config
        self.model = None
        self.processor = None
        
    def load_model(self):
        """Load MMS model with Twi language adapter."""
        print(f"🚀 Loading MMS model: {self.config.model_name}")
        
        # Load processor (tokenizer + feature extractor)
        self.processor = AutoProcessor.from_pretrained(
            self.config.model_name,
            cache_dir=self.config.cache_dir
        )
        
        # Set the target language
        self.processor.tokenizer.set_target_lang(self.config.adapter_lang)
        
        # Load model
        self.model = Wav2Vec2ForCTC.from_pretrained(
            self.config.model_name,
            cache_dir=self.config.cache_dir,
            ignore_mismatched_sizes=True,
            mask_time_prob=self.config.mask_time_prob,
            layerdrop=self.config.layerdrop,
            # Set vocabulary size for Twi
            vocab_size=len(self.processor.tokenizer)
        )
        
        # Load language-specific adapter
        self.model.load_adapter(self.config.adapter_lang)
        
        # Freeze feature encoder if specified
        if self.config.freeze_feature_encoder:
            self.model.freeze_feature_encoder()
            print("❄️ Feature encoder frozen")
        
        # Enable gradient checkpointing
        if self.config.gradient_checkpointing:
            self.model.gradient_checkpointing_enable()
            print("✅ Gradient checkpointing enabled")
        
        print(f"✅ Model loaded with Twi adapter")
        print(f"📊 Model parameters: {self.model.num_parameters():,}")
        print(f"🔤 Vocabulary size: {len(self.processor.tokenizer)}")
        
        return self.model, self.processor
    
    def get_twi_alphabet(self) -> List[str]:
        """Get Twi-specific alphabet including special characters."""
        # Standard English alphabet
        alphabet = list("abcdefghijklmnopqrstuvwxyz")
        
        # Add Twi-specific characters
        twi_chars = ['ɛ', 'ɔ', 'Ɛ', 'Ɔ']
        alphabet.extend(twi_chars)
        
        # Add uppercase
        alphabet.extend([c.upper() for c in "abcdefghijklmnopqrstuvwxyz"])
        
        # Add special tokens
        alphabet.extend([' ', "'", '-', '.', ',', '?', '!'])
        
        return alphabet

# Load model
model_loader = MMSModelLoader(config)
model, processor = model_loader.load_model()

In [ ]:
# Data preparation for MMS
class MMSDataProcessor:
    """Process data for MMS training."""
    
    def __init__(self, processor, config: MMSConfig):
        self.processor = processor
        self.config = config
        
    def load_twi_data(self) -> DatasetDict:
        """Load Twi datasets."""
        datasets_list = []
        
        # Check for processed data
        metadata_path = self.config.data_dir / "metadata/all_datasets_processed.csv"
        if metadata_path.exists():
            print("📊 Loading processed Twi data...")
            df = pd.read_csv(metadata_path)
            
            # Handle different column names
            if 'audio_path' in df.columns:
                text_col = 'processed_text' if 'processed_text' in df.columns else 'original_text'
                
                if text_col in df.columns:
                    df = df.dropna(subset=['audio_path', text_col])
                    
                    # Find valid audio files
                    valid_samples = []
                    for idx, row in df.iterrows():
                        audio_path = row['audio_path']
                        
                        # Try different path patterns
                        possible_paths = [
                            Path(audio_path) if Path(audio_path).is_absolute() else self.config.data_dir / audio_path,
                            self.config.data_dir / 'raw' / 'common_voice_twi' / 'extracted' / 'tw_validated_0' / 'tw_validated_0' / Path(audio_path).name,
                            self.config.data_dir / 'audio' / Path(audio_path).name,
                        ]
                        
                        for path in possible_paths:
                            if path.exists():
                                valid_samples.append({
                                    'audio': str(path),
                                    'text': row[text_col]
                                })
                                break
                    
                    if valid_samples:
                        dataset = Dataset.from_pandas(pd.DataFrame(valid_samples))
                        datasets_list.append(dataset)
                        print(f"✅ Loaded {len(dataset)} samples from processed data")
        
        # Load Common Voice if available
        cv_path = self.config.data_dir / "processed/common_voice_twi_final"
        if cv_path.exists():
            print("📻 Loading Common Voice data...")
            cv_data = self._load_common_voice(cv_path)
            if cv_data:
                datasets_list.append(cv_data)
        
        if datasets_list:
            # Combine datasets
            from datasets import concatenate_datasets
            combined = concatenate_datasets(datasets_list)
            
            # Create splits
            return self._create_splits(combined)
        else:
            print("⚠️ No real data found, creating dummy dataset for testing")
            return self._create_dummy_data()
    
    def _load_common_voice(self, path: Path) -> Optional[Dataset]:
        """Load Common Voice dataset."""
        data = []
        audio_files = list(path.glob("**/*.wav")) + list(path.glob("**/*.mp3"))
        
        for audio_file in audio_files[:500]:  # Limit for testing
            txt_file = audio_file.with_suffix('.txt')
            if txt_file.exists():
                with open(txt_file, 'r', encoding='utf-8') as f:
                    text = f.read().strip()
                data.append({'audio': str(audio_file), 'text': text})
        
        if data:
            print(f"   Found {len(data)} Common Voice samples")
            return Dataset.from_pandas(pd.DataFrame(data))
        return None
    
    def _create_splits(self, dataset: Dataset) -> DatasetDict:
        """Create train/val/test splits."""
        dataset = dataset.shuffle(seed=42)
        
        # 80/10/10 split
        train_size = int(0.8 * len(dataset))
        val_size = int(0.1 * len(dataset))
        
        train_dataset = dataset.select(range(train_size))
        val_dataset = dataset.select(range(train_size, train_size + val_size))
        test_dataset = dataset.select(range(train_size + val_size, len(dataset)))
        
        return DatasetDict({
            'train': train_dataset,
            'validation': val_dataset,
            'test': test_dataset
        })
    
    def _create_dummy_data(self) -> DatasetDict:
        """Create dummy dataset for testing."""
        # Create dummy audio files
        dummy_dir = self.config.data_dir / "dummy_audio"
        dummy_dir.mkdir(parents=True, exist_ok=True)
        
        dummy_data = []
        for i, text in enumerate([
            "Wo ho te sɛn",
            "Me ma wo akye",
            "Ɛyɛ sɛn",
            "Me da wo ase",
            "Nante yie"
        ]):
            # Create dummy audio file
            audio_file = dummy_dir / f"dummy_{i}.wav"
            if not audio_file.exists():
                # Generate 1 second of silence
                dummy_audio = np.zeros(16000, dtype=np.float32)
                sf.write(audio_file, dummy_audio, 16000)
            
            dummy_data.append({"text": text, "audio": str(audio_file)})
        
        dataset = Dataset.from_pandas(pd.DataFrame(dummy_data))
        return DatasetDict({
            'train': dataset,
            'validation': dataset,
            'test': dataset
        })
    
    def prepare_dataset(self, batch):
        """Prepare dataset for MMS training - FIXED version."""
        audio_arrays = []
        
        for audio_path in batch["audio"]:
            if Path(audio_path).exists():
                # Load audio
                audio, sr = librosa.load(audio_path, sr=self.config.sample_rate)
                
                # Limit duration
                max_samples = int(self.config.max_duration * self.config.sample_rate)
                if len(audio) > max_samples:
                    audio = audio[:max_samples]
                
                audio_arrays.append(audio)
            else:
                # Dummy audio for missing files
                audio_arrays.append(np.zeros(self.config.sample_rate, dtype=np.float32))
        
        # Process audio - DON'T pass padding here
        batch["input_values"] = self.processor(
            audio_arrays,
            sampling_rate=self.config.sample_rate,
            return_tensors="pt"
        ).input_values
        
        # Process text labels - use tokenizer directly
        with self.processor.as_target_processor():
            batch["labels"] = self.processor.tokenizer(
                batch["text"],
                return_tensors="pt"
            ).input_ids
        
        return batch

# Load and prepare data
data_processor = MMSDataProcessor(processor, config)
datasets = data_processor.load_twi_data()

print(f"\n📊 Dataset Statistics:")
for split, dataset in datasets.items():
    print(f"  {split}: {len(dataset)} samples")

# Prepare datasets - with error handling
print("\n🔄 Preparing datasets for training...")
try:
    prepared_datasets = datasets.map(
        data_processor.prepare_dataset,
        batched=True,
        batch_size=8,
        remove_columns=datasets["train"].column_names
    )
    print("✅ Data preparation complete!")
except Exception as e:
    print(f"⚠️ Error during preparation: {e}")
    print("Using fallback individual processing...")
    
    # Fallback: process individually
    train_data = []
    for item in datasets["train"]:
        try:
            if Path(item["audio"]).exists():
                audio, _ = librosa.load(item["audio"], sr=16000)
            else:
                audio = np.zeros(16000, dtype=np.float32)
            
            inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
            
            with processor.as_target_processor():
                labels = processor.tokenizer(item["text"], return_tensors="pt")
            
            train_data.append({
                "input_values": inputs.input_values[0],
                "labels": labels.input_ids[0]
            })
        except:
            pass
    
    if train_data:
        prepared_datasets = DatasetDict({
            "train": Dataset.from_list(train_data),
            "validation": Dataset.from_list(train_data[:5]),
            "test": Dataset.from_list(train_data[:5])
        })
        print(f"✅ Fallback processing complete with {len(train_data)} samples")

In [ ]:
# Custom data collator for MMS
from dataclasses import dataclass
from typing import Dict, List, Union
import torch

@dataclass
class DataCollatorCTCWithPadding:
    """Data collator for CTC models."""
    
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Separate inputs and labels
        input_features = [{"input_values": feature["input_values"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )
        
        with self.processor.as_target_processor():
            labels_batch = self.processor.pad(
                label_features,
                padding=self.padding,
                return_tensors="pt",
            )
        
        # Replace padding with -100 to ignore in loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        batch["labels"] = labels
        
        return batch

# Initialize data collator
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

In [ ]:
# Training setup
class MMSTrainer:
    """MMS model trainer."""
    
    def __init__(self, model, processor, config: MMSConfig):
        self.model = model
        self.processor = processor
        self.config = config
        self.trainer = None
        
    def compute_metrics(self, eval_pred):
        """Compute WER and CER metrics."""
        pred_logits = eval_pred.predictions
        pred_ids = np.argmax(pred_logits, axis=-1)
        
        # Replace -100 with pad token
        eval_pred.label_ids[eval_pred.label_ids == -100] = self.processor.tokenizer.pad_token_id
        
        # Decode predictions and references
        pred_str = self.processor.batch_decode(pred_ids)
        label_str = self.processor.batch_decode(eval_pred.label_ids, group_tokens=False)
        
        # Calculate metrics
        wer = evaluate.load("wer")
        cer = evaluate.load("cer")
        
        wer_score = wer.compute(predictions=pred_str, references=label_str)
        cer_score = cer.compute(predictions=pred_str, references=label_str)
        
        return {"wer": wer_score, "cer": cer_score}
    
    def setup_training(self, train_dataset, eval_dataset, data_collator):
        """Setup training arguments and trainer."""
        
        training_args = TrainingArguments(
            output_dir=str(self.config.output_dir),
            group_by_length=True,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            eval_strategy="steps",  # FIXED: Changed from evaluation_strategy
            save_strategy="steps",
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            logging_steps=self.config.logging_steps,
            learning_rate=self.config.learning_rate,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            save_total_limit=2,
            load_best_model_at_end=True,
            metric_for_best_model="wer",
            greater_is_better=False,
            fp16=self.config.fp16,
            gradient_checkpointing=self.config.gradient_checkpointing,
            push_to_hub=False,
            report_to=["tensorboard"],
            remove_unused_columns=False,
        )
        
        self.trainer = Trainer(
            model=self.model,
            data_collator=data_collator,
            args=training_args,
            compute_metrics=self.compute_metrics,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            tokenizer=self.processor.feature_extractor,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
        )
        
        print("✅ Trainer initialized!")
        
    def train(self):
        """Start training."""
        print(f"\n🏋️ Starting MMS training...")
        print(f"📊 Training on {torch.cuda.device_count()} GPU(s)")
        
        # Train
        train_result = self.trainer.train()
        
        # Save model
        self.trainer.save_model()
        self.processor.save_pretrained(self.config.output_dir)
        
        # Save metrics
        with open(self.config.output_dir / "training_results.json", "w") as f:
            json.dump(train_result.metrics, f, indent=2)
        
        print("✅ Training complete!")
        return train_result

# Check if prepared_datasets exists and has data
if 'prepared_datasets' in locals() and len(prepared_datasets["train"]) > 0:
    # Setup trainer
    mms_trainer = MMSTrainer(model, processor, config)
    mms_trainer.setup_training(
        prepared_datasets["train"],
        prepared_datasets["validation"],
        data_collator
    )
    print(f"🎯 Ready to train with {len(prepared_datasets['train'])} samples!")
    print("Uncomment the next line to start training:")
    print("# train_result = mms_trainer.train()")
else:
    print("⚠️ No prepared datasets available. Please check data loading.")

In [ ]:
# Evaluation
class MMSEvaluator:
    """Evaluate MMS model performance."""
    
    def __init__(self, model_path: Path, test_dataset, processor):
        self.model_path = model_path
        self.test_dataset = test_dataset
        self.processor = processor
        self.model = None
        
    def load_model(self):
        """Load trained model."""
        self.model = Wav2Vec2ForCTC.from_pretrained(self.model_path)
        self.model.eval()
        
        if torch.cuda.is_available():
            self.model = self.model.cuda()
        
        print(f"✅ Model loaded from {self.model_path}")
        
    def evaluate(self) -> Dict:
        """Evaluate model on test set."""
        print("\n📊 Evaluating MMS model...")
        
        predictions = []
        references = []
        inference_times = []
        
        for item in tqdm(self.test_dataset):
            # Load audio
            if Path(item["audio"]).exists():
                audio, _ = librosa.load(item["audio"], sr=16000)
            else:
                audio = np.zeros(16000, dtype=np.float32)
            
            # Process audio
            inputs = self.processor(
                audio,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )
            
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}
            
            # Inference
            import time
            start = time.time()
            
            with torch.no_grad():
                logits = self.model(**inputs).logits
            
            inference_time = time.time() - start
            inference_times.append(inference_time)
            
            # Decode
            predicted_ids = torch.argmax(logits, dim=-1)
            transcription = self.processor.batch_decode(predicted_ids)[0]
            
            predictions.append(transcription)
            references.append(item["text"])
        
        # Calculate metrics
        wer_metric = evaluate.load("wer")
        cer_metric = evaluate.load("cer")
        
        wer = wer_metric.compute(predictions=predictions, references=references)
        cer = cer_metric.compute(predictions=predictions, references=references)
        
        results = {
            "wer": wer,
            "cer": cer,
            "avg_inference_time": np.mean(inference_times),
            "total_samples": len(predictions),
            "sample_predictions": list(zip(references[:5], predictions[:5]))
        }
        
        print(f"\n📊 Evaluation Results:")
        print(f"  WER: {wer:.2%}")
        print(f"  CER: {cer:.2%}")
        print(f"  Avg inference time: {np.mean(inference_times):.3f}s")
        
        # Show sample predictions
        print(f"\n📝 Sample Predictions:")
        for ref, pred in results["sample_predictions"]:
            print(f"  Reference: {ref}")
            print(f"  Predicted: {pred}")
            print()
        
        return results
    
    def benchmark_languages(self):
        """Benchmark performance across different language adapters."""
        print("\n🌍 Benchmarking language adapters...")
        
        # Languages to test (related Akan languages and dialects)
        # aka = Akan (includes Twi and Fante)
        # ewe = Ewe (related Kwa language)
        # gaa = Ga (another Ghana language)
        test_languages = ['aka', 'ewe', 'gaa']  # Languages available in MMS for Ghana region
        
        results = {}
        
        for lang in test_languages:
            try:
                print(f"\n  Testing {lang}...")
                
                # Check if language adapter exists
                if lang in self.processor.tokenizer.vocab:
                    # Load adapter
                    self.processor.tokenizer.set_target_lang(lang)
                    self.model.load_adapter(lang)
                    
                    # Test on a few samples
                    test_samples = self.test_dataset.select(range(min(10, len(self.test_dataset))))
                    
                    predictions = []
                    for item in test_samples:
                        if Path(item["audio"]).exists():
                            audio, _ = librosa.load(item["audio"], sr=16000)
                        else:
                            audio = np.zeros(16000, dtype=np.float32)
                        
                        inputs = self.processor(audio, sampling_rate=16000, return_tensors="pt")
                        
                        if torch.cuda.is_available():
                            inputs = {k: v.cuda() for k, v in inputs.items()}
                        
                        with torch.no_grad():
                            logits = self.model(**inputs).logits
                        
                        predicted_ids = torch.argmax(logits, dim=-1)
                        transcription = self.processor.batch_decode(predicted_ids)[0]
                        predictions.append(transcription)
                    
                    results[lang] = predictions
                    print(f"    ✅ {lang}: Success")
                else:
                    print(f"    ⚠️ {lang}: Not available in MMS")
                    results[lang] = None
                
            except Exception as e:
                print(f"    ❌ {lang}: {str(e)}")
                results[lang] = None
        
        return results

# Run evaluation (after training)
# evaluator = MMSEvaluator(
#     config.output_dir,
#     datasets["test"],
#     processor
# )
# evaluator.load_model()
# results = evaluator.evaluate()

In [ ]:
# Interactive testing
class MMSInteractiveTester:
    """Interactive testing for MMS model."""
    
    def __init__(self, model, processor):
        self.model = model
        self.processor = processor
        self.model.eval()
        
        if torch.cuda.is_available():
            self.model = self.model.cuda()
    
    def transcribe(self, audio_path: str) -> str:
        """Transcribe audio file."""
        # Load audio
        audio, sr = librosa.load(audio_path, sr=16000)
        
        # Process
        inputs = self.processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )
        
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        
        # Inference
        with torch.no_grad():
            logits = self.model(**inputs).logits
        
        # Decode
        predicted_ids = torch.argmax(logits, dim=-1)
        transcription = self.processor.batch_decode(predicted_ids)[0]
        
        return transcription
    
    def test_twi_phrases(self):
        """Test common Twi phrases."""
        print("\n🎤 Testing Twi phrases (would need audio files):")
        
        test_phrases = [
            ("greeting_morning.wav", "Me ma wo akye"),
            ("how_are_you.wav", "Wo ho te sɛn"),
            ("thank_you.wav", "Me da wo ase"),
            ("goodbye.wav", "Nante yie"),
        ]
        
        for audio_file, expected in test_phrases:
            audio_path = Path(f"../data/test/{audio_file}")
            if audio_path.exists():
                transcription = self.transcribe(str(audio_path))
                print(f"\n  Expected: {expected}")
                print(f"  Got: {transcription}")
                print(f"  Match: {'✅' if transcription.lower() == expected.lower() else '❌'}")

# Test model interactively
tester = MMSInteractiveTester(model, processor)
# tester.test_twi_phrases()

## Summary

This notebook implements Meta's MMS (Massively Multilingual Speech) model for Twi ASR:

### Key Features:
1. **Pre-trained Multilingual Model**: Leverages MMS-1B with 1,100+ language support
2. **Language Adapters**: Uses Twi-specific adapter for better performance
3. **CTC Training**: Implements Connectionist Temporal Classification
4. **Efficient Fine-tuning**: Freezes feature encoder, only trains upper layers
5. **Multi-metric Evaluation**: Tracks both WER and CER

### Advantages of MMS:
- Pre-trained on massive multilingual data
- Native support for Twi language
- Smaller model size compared to Whisper
- Good for low-resource languages
- Language adapter architecture allows easy switching

### Training Tips:
- Start with frozen feature encoder
- Use smaller learning rate (3e-4)
- Shorter audio segments work better (< 20s)
- Monitor both WER and CER metrics

### Next Steps:
1. Prepare more Twi training data
2. Fine-tune hyperparameters
3. Compare with Whisper performance
4. Test on real-world Twi audio